In [1]:
import hail as hl
from ukb_utils import hail_init
import numpy as np
import os

from ko_utils import ko
from ko_utils import io
from ko_utils import samples
from ko_utils.ko import PLOF_CSQS, MISSENSE_CSQS, SYNONYMOUS_CSQS, OTHER_CSQS
import scipy.stats as stats
from ukb_utils import variants

In [2]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh38')

Running on Apache Spark version 3.1.3
SparkUI available at http://compa039.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.78-b17627756568
LOGGING: writing to logs/hail/hail_test_export.log


In [3]:
#hl._set_flags(no_whole_stage_codegen='1')

In [3]:
hl.__version__

'0.2.78-b17627756568'

In [7]:
input_path = "data/conditional/rare/combined/ukb_eur_wes_200k_chr21_maf0to5e-2_pLoF_damaging_missense.mt"
input_type = "mt"
pheno_file = "data/phenotypes/curated_covar_phenotypes_binary_200k.tsv"
phenotypes = "BC_combined,BC_combined_primary_care"
covariates = "PC1,PC2"

In [5]:
main = hl.read_matrix_table(path)

In [6]:
ht = hl.import_table(pheno_file,
                 types={'eid': hl.tstr},
                 missing=["",'""',"NA"],
                 impute=True,
                 force=True,
                 key='eid'
                 )
main = main.annotate_cols(pheno=ht[main.s])

2022-07-31 22:39:18 Hail: INFO: Reading table to impute column types
2022-07-31 22:39:26 Hail: INFO: Loading 118 fields. Counts by type:
  bool: 71
  float64: 40
  int32: 6
  str: 1


In [19]:
main = main.annotate_rows(stats = hl.struct())

In [21]:
for phenotype in phenotypes:
    print(phenotype)
    
    mt = main
    
    # filter columns based on phenotype availability
    mt = mt.filter_cols(
        hl.is_defined(mt.pheno[phenotype])
    )
    
    # filter columns based on covariate availability
    mt = mt.filter_cols(
        hl.is_defined([mt.pheno[x] for x in covariates])
    )
    
    # get allele count
    mt = mt.annotate_rows(
        AC = hl.agg.sum(mt.DS)
    )
    
    # annotate main matrix table with allele counts
    main = main.annotate_rows(
        stats = main.stats.annotate(
            AC = mt.rows()[main.row_key].AC
        )
    )
    
    main = main.annotate_rows(
        stats = main.stats.rename({'AC': "ac_" + str(phenotype)})
    )
    
     
    

BC_combined
BC_combined_primary_care


In [8]:
main = io.import_table(input_path, input_type, calc_info = False)

# setup phenotypes
ht = hl.import_table(pheno_file,
             types={'eid': hl.tstr},
             missing=["",'""',"NA"],
             impute=True,
             force=True,
             key='eid'
             )



# read in covariates
phenotypes = phenotypes.strip().split(',')
split_covariates = covariates.strip().split(',')



2022-07-31 22:40:41 Hail: INFO: Reading table to impute column types
2022-07-31 22:40:47 Hail: INFO: Loading 118 fields. Counts by type:
  bool: 71
  float64: 40
  int32: 6
  str: 1


In [10]:
split_covariates

['PC1', 'PC2']

In [15]:
# this is the table we keep track of
main = main.annotate_cols(pheno=ht[main.s])
main = main.annotate_rows(stats = hl.struct())

for phenotype in phenotypes:

    # point to original
    mt = main

    # need build covaraites in context of mt
    columns = [mt.pheno[x] for x in split_covariates]
    columns += [mt.pheno[phenotype]]
    
    # filter columns based on phenotype availability
    mt = mt.filter_cols(
        hl.is_defined(mt.pheno[phenotype])
    )
    # filter columns based on covariate availability
    #mt = mt.filter_cols(
    #    hl.is_defined(covariates)
    #)
    
    
    # get allele count
    mt = mt.annotate_rows(
        AC = hl.agg.sum(mt.DS)
    )
    # annotate main matrix table with allele counts
    main = main.annotate_rows(
        stats = main.stats.annotate(
            AC = mt.rows()[main.row_key].AC
        )
    )
    main = main.annotate_rows(
        stats = main.stats.rename({'AC': "ac_" + str(phenotype)})
    )

In [26]:
mt = mt.annotate_entries(GT = ds_to_fake_gt(mt.DS))

In [28]:
def DS_to_fake_GT(DS):
    return (hl.case()
            .when(DS == 0, hl.parse_call("0/0"))
            .when(DS == 1, hl.parse_call("1/0"))
            .when(DS == 2, hl.parse_call("1/1"))
            .or_missing()
           )
    

In [3]:
mt = hl.balding_nichols_model(3, 100, 100, reference_genome='GRCh37')

2022-07-31 23:15:03 Hail: INFO: balding_nichols_model: generating genotypes for 3 populations, 100 samples, and 100 variants...


In [4]:
pruned_varaints = hl.ld_prune(mt.GT, r2=0.2, bp_window_size=500)

2022-07-31 23:15:32 Hail: INFO: ld_prune: running local pruning stage with max queue size of 894785 variants
2022-07-31 23:15:34 Hail: INFO: Coerced sorted dataset
2022-07-31 23:15:35 Hail: INFO: wrote table with 99 rows in 8 partitions to /tmp/NzZnEFcEY0Vu7BVG8LzUuO
    Total size: 3.18 KiB
    * Rows: 3.17 KiB
    * Globals: 11.00 B
    * Smallest partition: 11 rows (375.00 B)
    * Largest partition:  13 rows (424.00 B)
2022-07-31 23:15:35 Hail: INFO: Coerced sorted dataset
2022-07-31 23:15:36 Hail: INFO: Wrote all 1 blocks of 99 x 100 matrix with block size 4096.
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/well/lindgren/users/mmq446/conda/skylake/envs/hail-v0.2.95/lib/python3.6/site-packages/py4j/java_gateway.py", line 1207, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceba

Py4JError: An error occurred while calling o103.executeEncode

In [ ]:
pruned_varaints.count()

In [26]:
hl.parse_call("").show()

""
""
call
1/1
